In [ ]:
from datetime import datetime
import os
from pathlib import Path

import gymnasium as gym
import gym_pusht
import numpy as np
import pygame
import imageio.v2 as imageio
from pathlib import Path
import torch

In [ ]:
mode = "obstacle"
data_path = Path(os.environ.get("PUSHT_OUTPUT_DIR", "outputs/pusht-gym")) / f"{mode}_mouse"
run_path = data_path / datetime.now().strftime("run_%Y%m%d_%H%M%S")
video_path = run_path / "obses"
video_path.mkdir(parents=True, exist_ok=True)

env = gym.make("gym_pusht/PushT-v0", 
               render_mode="human", 
               obs_type="state",
               scale=30,
               damping=None,
            #    tee_color="LightSkyBlue",
                enable_obstacle=True,
               )
base_env = env.unwrapped

recording_fps = 10
steps = 200
episodes = 4
startup_delay_seconds = 1.0
startup_ramp_steps = 5

abs_actions = np.zeros([episodes, steps, 2], dtype=np.float32)
rel_actions = np.zeros([episodes, steps, 2], dtype=np.float32)
states = np.zeros([episodes, steps, 5], dtype=np.float32)
lengths = np.zeros(episodes, dtype=np.int64)
loop_clock = pygame.time.Clock()


for episode in range(episodes):
    observation, info = env.reset()

    # Give the operator time to place the cursor after reset, then ramp from
    # the agent's reset position to the cursor target to avoid a sudden jump.
    delay_frames = int(startup_delay_seconds * recording_fps)
    for _ in range(delay_frames):
        env.render()
        pygame.event.pump()
        loop_clock.tick(recording_fps)
    initial_action = observation[:2].copy()

    episode_video_path = video_path / f"episode_{episode:03d}.mp4"

    with imageio.get_writer(
            episode_video_path,
            fps=recording_fps, 
            codec="libx264",
            is_batch=False
        ) as video:
        for step in range(steps):
            env.render()
            mouse_x, mouse_y = pygame.mouse.get_pos()
            mouse_action = np.array([mouse_x, mouse_y], dtype=np.float32)
            if step < startup_ramp_steps:
                ramp = (step + 1) / startup_ramp_steps
                abs_action = initial_action + ramp * (mouse_action - initial_action)
            else:
                abs_action = mouse_action

            # Capture the scene drawn by the environment.
            frame = base_env._get_img(
                base_env._draw(), width=224, height=224, render_action=False
            )
            video.append_data(frame)

            # Store actions and states
            abs_actions[episode,step,:] = abs_action
            states[episode,step,:] = observation
            rel_actions[episode,step,:] = abs_action - observation[:2]

            observation, reward, terminated, truncated, info = env.step(abs_action)

            lengths[episode] = step + 1
            # Keep mouse demonstrations at the environment's 10 Hz action rate.
            loop_clock.tick(recording_fps)
            if terminated or truncated:
                break
env.close()

# Convert rel_actions and states to torch.tensor and save as pth file
torch.save(torch.from_numpy(abs_actions), run_path / "abs_actions.pth")
torch.save(torch.from_numpy(rel_actions), run_path / "rel_actions.pth")
torch.save(torch.from_numpy(states), run_path / "states.pth")
torch.save(torch.from_numpy(lengths), run_path / "seq_lengths.pth")


In [ ]:
from IPython.display import display, Video, Markdown
for episode in range(episodes):
    display(Markdown(f"### Episode: {episode:03d} Mode: {mode} "))
    display(Video(filename=video_path / f"episode_{episode:03d}.mp4", embed=True, width=300, html_attributes='controls loop muted playsinline style="max-width:100%;height:auto"'))